# K-Means & Hierarchical Clustering on `sales_data_sample.csv`
The number of clusters is chosen with the **elbow method** (cross-checked with silhouette scores).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set()

## 1. Load the data

In [ ]:
df = pd.read_csv('sales_data_sample.csv', sep=',', encoding='Latin-1')
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## 2. Preprocessing
- Drop identifier / contact / address columns (no clustering value; many are mostly null).
- Drop `ORDERDATE` (month, quarter and year are already present), `STATUS`, `COUNTRY` (too many levels) and `PRODUCTCODE` (an ID; label-encoding it would create a meaningless order).
- One-hot encode `PRODUCTLINE` and `DEALSIZE`.
- **Standardise all features**: K-Means and hierarchical clustering are distance-based, so unscaled columns like `SALES` would dominate.

In [ ]:
drop_cols = ['ORDERNUMBER', 'ORDERDATE', 'STATUS', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE',
             'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY',
             'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME']
data = df.drop(columns=[c for c in drop_cols if c in df.columns])

data = pd.get_dummies(data, columns=['PRODUCTLINE', 'DEALSIZE'], dtype=int)
data = data.dropna()
print(data.shape)
data.dtypes

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(data)
X.shape

## 3. Elbow method

In [ ]:
K = range(1, 11)
inertias = []
for k in K:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K, inertias, 'bx-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia (within-cluster sum of squares)')
plt.title('Elbow method')
plt.xticks(list(K))
plt.show()

In [ ]:
# Locate the elbow programmatically: the point farthest from the line joining the first and last points
x = np.array(list(K), dtype=float)
y = np.array(inertias)
p1, p2 = np.array([x[0], y[0]]), np.array([x[-1], y[-1]])
pts = np.column_stack([x, y])
v = p2 - p1
d = np.abs(v[0] * (p1[1] - pts[:, 1]) - v[1] * (p1[0] - pts[:, 0])) / np.linalg.norm(v)
k_elbow = int(x[np.argmax(d)])
print('Elbow detected at k =', k_elbow)

In [ ]:
# Cross-check with silhouette score (higher = better separated clusters)
sil = []
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    sil.append(silhouette_score(X, labels))

plt.figure(figsize=(10, 4))
plt.plot(range(2, 11), sil, 'go-')
plt.xlabel('k'); plt.ylabel('Silhouette score'); plt.title('Silhouette analysis')
plt.show()

## 4. K-Means with the chosen k
`K_OPTIMAL` defaults to the detected elbow. Change it manually if your plot suggests a different value.

In [ ]:
K_OPTIMAL = k_elbow
kmeans = KMeans(n_clusters=K_OPTIMAL, n_init=10, random_state=42)
data['KMeans_Cluster'] = kmeans.fit_predict(X)
data['KMeans_Cluster'].value_counts().sort_index()

### Visualise clusters (PCA to 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
centers_pca = pca.transform(kmeans.cluster_centers_)
print('Variance explained by 2 PCs:', pca.explained_variance_ratio_.sum().round(3))

plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=data['KMeans_Cluster'], palette='Set1', s=30)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='x', s=250, label='Centroids')
plt.xlabel('PCA1'); plt.ylabel('PCA2'); plt.title(f'K-Means clusters (k={K_OPTIMAL})')
plt.legend()
plt.show()

### Cluster profiles (mean of original features)

In [ ]:
num_cols = ['QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'MSRP']
num_cols = [c for c in num_cols if c in data.columns]
data.groupby('KMeans_Cluster')[num_cols].mean().round(2)

In [ ]:
dealsize_cols = [c for c in data.columns if c.startswith('DEALSIZE_')]
data.groupby('KMeans_Cluster')[dealsize_cols].mean().round(2)

## 5. Hierarchical (agglomerative) clustering

In [ ]:
Z = linkage(X, method='ward')
plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30, show_leaf_counts=True)
plt.title('Dendrogram (Ward linkage, truncated)')
plt.xlabel('Cluster size'); plt.ylabel('Distance')
plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage='ward')
data['Hier_Cluster'] = agg.fit_predict(X)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=data['Hier_Cluster'], palette='Set2', s=30)
plt.xlabel('PCA1'); plt.ylabel('PCA2'); plt.title(f'Hierarchical clusters (k={K_OPTIMAL})')
plt.show()

print('Silhouette  K-Means     :', round(silhouette_score(X, data['KMeans_Cluster']), 3))
print('Silhouette  Hierarchical:', round(silhouette_score(X, data['Hier_Cluster']), 3))